In [ ]:
# Bước 1: Set seed cố định
import os, torch, random
import numpy as np

os.environ['PYTHONUTF8'] = '1'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [ ]:
# Bước 2: Nạp mô hình và các tài nguyên
import sys
from pathlib import Path

CODE_DIR = Path("/code") if Path("/code/predict.py").exists() else Path.cwd()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from predict import load_models
from lifemate.agent import LifeMateAgent

# Không dùng API ngoài: nạp trực tiếp Dual-ASR và Qwen3-4B từ /code/saved_models/
asr_model, model, model_load_s = load_models(
    asr_mode="transformers",
    llm_backend="transformers",
    dtype="bfloat16",
    max_new_tokens=384,
    seed=42,
)

In [ ]:
# Bước 3: Đọc nội dung test cases
import json, sys
from pathlib import Path

ROOT = Path("/data")
SPLIT = next(p.name for p in sorted(ROOT.iterdir()) if p.is_dir() and (p / f"{p.name}_tools_catalog.json").exists())
sys.path.insert(0, str(ROOT / SPLIT))
if (ROOT / "zaic_tools").exists():
    sys.path.insert(0, str(ROOT))
from zaic_tools import ToolEnv

catalog = json.load(open(ROOT / SPLIT / f"{SPLIT}_tools_catalog.json", encoding="utf-8"))
agent = LifeMateAgent(llm=model, catalog=catalog, max_steps=10, max_new_tokens=384)

with open(ROOT / SPLIT / f"{SPLIT}.jsonl", encoding="utf-8") as f:
    test_cases = [json.loads(line) for line in f if line.strip()]

In [ ]:
# Bước 4: Thực hiện dự đoán, in kết quả và thời gian
import time
all_result = []
split_dir = ROOT / SPLIT

for item in test_cases:
    t1 = time.perf_counter()
    env = ToolEnv(SPLIT, item["id"], root=ROOT)
    deadline = t1 + 25.0 - 2.5

    try:
        transcript_map = asr_model.transcribe_sample(item, split_dir, history_dual=False)
    except Exception as e:
        print(f"[ASR ERROR] {item['id']}: {e}")
        transcript_map = {}

    try:
        agent.run_sample(env, item, transcript_map, deadline=deadline)
    except Exception as e:
        print(f"[AGENT ERROR] {item['id']}: {e}")

    tool_calls, state_hash = env.get_tool_calls(), env.get_state_hash()
    ms = int((time.perf_counter() - t1) * 1000)
    all_result.append({"id": item["id"], "tool_calls": tool_calls, "state_hash": state_hash, "time": ms})

Path("/result").mkdir(parents=True, exist_ok=True)
with open("/result/jupyter_submission.jsonl", "w", encoding="utf-8") as f:
    for r in all_result:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

total_ms = sum(r["time"] for r in all_result)
print(f"Done! {len(all_result)} samples -> /result/jupyter_submission.jsonl | Total={total_ms/1000:.2f}s | Avg={total_ms/max(1,len(all_result)):.1f}ms/sample")